In [1]:
import torch
import clip
from PIL import Image
import os
import torchvision.transforms as T
from tqdm import tqdm
from train_minitheia import TheiaStudent  # Ajusta si tu archivo tiene otro nombre
import numpy as np
import matplotlib.pyplot as plt
import timm
import torch.nn as nn
import random
import warnings
warnings.filterwarnings('ignore')

from train_minitheia import TheiaStudent 

/home/sergio/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -----------------------------
# CONFIG
# -----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_WEIGHTS = "runs/theia_mini/theia_mini_best.pt"
N_SAMPLES = 0

# Imágenes de prueba
IMAGE_PATHS = [
    "perro.jpg",
    "gato.jpg"
]

TEXT_PROMPTS = [
    "un perro corriendo en el césped",
    "un gato mirando a la cámara"
]

In [3]:
# -----------------------------
# CARGAR MODELOS
# -----------------------------
print("Cargando CLIP teacher...")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=DEVICE, jit=False)

print("Cargando Mini-Theia...")
student = TheiaStudent().to(DEVICE)
student.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=DEVICE))
student.eval()

to_tensor = T.Compose([
    T.Resize((224,224)),
    T.ToTensor()
])

# -----------------------------
# EXTRACCIÓN
# -----------------------------
def get_student_image_emb(path):
    img = Image.open(path).convert("RGB")
    tensor = to_tensor(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            clip_emb, dino_emb, fused = student(tensor)  # fused = 192 dims
    return clip_emb, fused

def get_clip_teacher_image_emb(path):
    img = Image.open(path).convert("RGB")
    tensor = clip_preprocess(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        image_features = clip_model.encode_image(tensor)
    return image_features

def get_clip_text_emb(text):
    with torch.no_grad():
        text_tokens = clip.tokenize(text).to(DEVICE)
        text_features = clip_model.encode_text(text_tokens)
    return text_features


# -----------------------------
# SIMILITUD
# -----------------------------
def cosine(a, b):
    return torch.nn.functional.cosine_similarity(a, b).item()

Cargando CLIP teacher...
Cargando Mini-Theia...


In [4]:
# -------- Retrieval Utility --------
def load_image(path):
    tf = T.Compose([
        T.Resize(256),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ])
    return tf(Image.open(path).convert("RGB"))

def cosine_similarity(a, b):
    return (a @ b.T).cpu().numpy()


def show_results(query_path, top_paths):
    fig, axs = plt.subplots(1, len(top_paths) + 1, figsize=(16, 4))

    # Query
    axs[0].imshow(Image.open(query_path))
    axs[0].set_title("Query")
    axs[0].axis("off")

    # Top-K
    for i, p in enumerate(top_paths):
        axs[i+1].imshow(Image.open(p))
        axs[i+1].set_title(f"Top {i+1}")
        axs[i+1].axis("off")

    plt.tight_layout()
    plt.show()

In [5]:
dataset_dir = "test"
k = 5

# Load model
model = TheiaStudent()
model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=DEVICE))
model = model.to(DEVICE).eval()

def testDINO(image, model):
    # Load query
    query_img = load_image(image).unsqueeze(0).to(DEVICE)
    
    # Compute query embedding
    with torch.no_grad():
        _, dino_query, _ = model(query_img)
    
    # Precompute gallery embeddings
    gallery_paths = []
    gallery_embeds = []
    
    all_image_paths = []
    for fname in os.listdir(dataset_dir):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            path = os.path.join(dataset_dir, fname)
            all_image_paths.append(path)
    if N_SAMPLES == 0:
        selected_paths = all_image_paths
    elif len(all_image_paths) > N_SAMPLES:
        selected_paths = random.sample(all_image_paths, N_SAMPLES) #
    else:
        selected_paths = all_image_paths
        print(f"Solo se encontraron {len(all_image_paths)} imágenes, procesando todas.")
    
    for path in tqdm(selected_paths):
        img = load_image(path).unsqueeze(0).to(DEVICE)
    
        with torch.no_grad():
            _, emb, _ = model(img)
    
        gallery_paths.append(path)
        gallery_embeds.append(emb.cpu().numpy()[0])
        
    # Compute similarity
    sims = gallery_embeds @ dino_query.cpu().numpy()[0]
    
    # Top-K
    top_idx = np.argsort(-sims)[:k]
    top_paths = [gallery_paths[i] for i in top_idx]
    # Show result
    show_results(image, top_paths)

In [6]:
# -----------------------------
# EVALUACIÓN
# -----------------------------
print("\n=== Evaluación CLIP texto ↔ imagen ===")

for text in TEXT_PROMPTS:
    print(f"\n⚫ Texto: {text}")

    teacher_text = get_clip_text_emb(text)

    best_img = None
    best_teacher_score = -1
    best_student_score = -1

    for img_path in IMAGE_PATHS:
        teacher_img = get_clip_teacher_image_emb(img_path)
        student_img, fused = get_student_image_emb(img_path)

        s_t = cosine(student_img, teacher_text)
        t_t = cosine(teacher_img, teacher_text)

        print(f"  Imagen: {img_path}")
        print(f"    Teacher CLIP score: {t_t:.4f}")
        print(f"    Student Mini-Theia score: {s_t:.4f}")

        if s_t > best_student_score:
            best_student_score = s_t
            best_img = img_path

    print(f"➡ Mini-Theia considera más similar: **{best_img}**")


=== Evaluación CLIP texto ↔ imagen ===

⚫ Texto: un perro corriendo en el césped
  Imagen: perro.jpg
    Teacher CLIP score: 0.2563
    Student Mini-Theia score: 0.2507
  Imagen: gato.jpg
    Teacher CLIP score: 0.1710
    Student Mini-Theia score: 0.2049
➡ Mini-Theia considera más similar: **perro.jpg**

⚫ Texto: un gato mirando a la cámara
  Imagen: perro.jpg
    Teacher CLIP score: 0.1799
    Student Mini-Theia score: 0.2294
  Imagen: gato.jpg
    Teacher CLIP score: 0.2520
    Student Mini-Theia score: 0.2343
➡ Mini-Theia considera más similar: **gato.jpg**


In [ ]:
testDINO("query.jpg", model)
testDINO("perro.jpg", model)
testDINO("gato.jpg", model)

 17%|█████████████████████████▏                                                                                                                         | 6866/40152 [00:43<03:20, 165.71it/s]